# nb10.3 - Mediation Done Right: When Adding a Control Makes the Bias Worse

*Week 10, Chapter 10.3 companion. Maya's referee asks: "Doesn't your fair-lending effect work *through*
credit-score formation? Show me the mediation analysis." Maya knows the request is reasonable. She
also knows that the textbook mediation recipe is full of traps. This notebook walks her through both
the right way and the wrong way to answer the question.*

The reveal-the-trick result: **adding a mediator as a control variable does not "make the effect
cleaner" - in general it biases your estimate.** Two cases:

1. **True mediator** $M$. The treatment $D$ changes $M$, and $M$ changes the outcome $Y$. Controlling
   for $M$ here estimates the *direct* effect ($D \to Y$ holding $M$ fixed), not the *total* effect.
   Reporting the direct-effect coefficient as the "treatment effect" understates the policy
   consequence. Worse, if there is unmeasured confounding between $M$ and $Y$, conditioning on $M$
   opens a non-causal pathway and biases the direct effect even further.

2. **Bad-control collider** $C$. The treatment $D$ and an unobserved confounder $U$ both cause $C$.
   $C$ has nothing to do with $Y$ on its own. Conditioning on $C$ opens a back-door from $D$ to $U$
   to $Y$. The "control" creates the bias you were trying to avoid.

We simulate both worlds with a known truth, decompose the bias, and show how the **sequential
ignorability** assumption (Imai, Keele & Yamamoto 2010, *Statistical Science* 25(1):51-71) sets up the
*conditions* under which a mediation decomposition is identified at all.

In [ ]:
import matplotlib
matplotlib.use("Agg")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

rng = np.random.default_rng(42)

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.float_format", lambda v: f"{v:0.4f}")

## 1. The clean mediator DGP (no unmeasured confounding)

A binary treatment $D$ is randomized at $\Pr(D=1) = 0.5$. A continuous mediator $M$ depends on $D$:
$$M = a \cdot D + u_M, \qquad u_M \sim \mathcal{N}(0, 1).$$
The outcome depends on **both** $D$ and $M$:
$$Y = c' \cdot D + b \cdot M + u_Y, \qquad u_Y \sim \mathcal{N}(0, 1).$$
The constants $a, b, c'$ are known to us. The **total effect** of $D$ on $Y$ is what an OLS regression
of $Y$ on $D$ alone identifies. By repeated substitution:
$$\mathbb{E}[Y \mid D=1] - \mathbb{E}[Y \mid D=0] = c' + a \cdot b.$$
The piece $a \cdot b$ is the **indirect (mediated) effect**: how much the treatment's effect on $M$,
amplified by $M$'s effect on $Y$, contributes to the total. The piece $c'$ is the **direct effect**:
how much $D$ changes $Y$ holding $M$ fixed. The classical Baron-Kenny decomposition (Baron & Kenny
1986, *Journal of Personality and Social Psychology* 51(6):1173-1182) says:
$$\text{total} = c' + a \cdot b = \text{direct} + \text{indirect}.$$
Let's verify on data.

In [ ]:
def make_mediator_dgp(rng, N=4000, a=0.8, b=0.7, c_prime=0.5):
    D = rng.binomial(1, 0.5, size=N)
    uM = rng.normal(0, 1, size=N)
    uY = rng.normal(0, 1, size=N)
    M = a * D + uM
    Y = c_prime * D + b * M + uY
    df = pd.DataFrame({"D": D, "M": M, "Y": Y})
    df.attrs["a"] = a; df.attrs["b"] = b; df.attrs["c_prime"] = c_prime
    return df

df = make_mediator_dgp(rng)
a, b, c_p = df.attrs["a"], df.attrs["b"], df.attrs["c_prime"]
print(f"truth: a={a}, b={b}, c_prime={c_p}")
print(f"       direct effect c'   = {c_p}")
print(f"       indirect effect ab = {a*b:0.3f}")
print(f"       total effect       = {c_p + a*b:0.3f}")

## 2. Baron-Kenny on the clean DGP

In [ ]:
def baron_kenny(df):
    # Step 1: total effect (Y ~ D)
    r1 = sm.OLS(df["Y"].values, sm.add_constant(df[["D"]].values)).fit()
    total = r1.params[1]
    # Step 2: mediator equation (M ~ D) gives a_hat
    r2 = sm.OLS(df["M"].values, sm.add_constant(df[["D"]].values)).fit()
    a_hat = r2.params[1]
    # Step 3: outcome with mediator (Y ~ D + M) gives c'_hat and b_hat
    r3 = sm.OLS(df["Y"].values, sm.add_constant(df[["D","M"]].values)).fit()
    c_prime_hat = r3.params[1]; b_hat = r3.params[2]
    indirect = a_hat * b_hat
    return {"total": total, "a_hat": a_hat, "b_hat": b_hat,
            "c_prime_hat": c_prime_hat, "indirect_ab": indirect,
            "direct + indirect": c_prime_hat + indirect}

bk = baron_kenny(df)
pd.Series(bk)

**Read the printout.** On a clean DGP with **no unmeasured confounding between $M$ and $Y$**,
Baron-Kenny recovers $a$, $b$, $c'$, and their sum. Indirect equals $\widehat a \cdot \widehat b$,
direct equals $\widehat{c'}$, and the two sum to the total. This is the textbook win - and it is also
the only setting in which Baron-Kenny *works*. We now stress it.

## 3. The same DGP but with $M \leftrightarrow Y$ confounding

In the real world the mediator and the outcome share unobserved causes. A neighborhood factor
$U$ raises both the borrower's credit score (the mediator $M$) and her income (the outcome $Y$).
If $U$ is unobserved and we control for $M$, we **open** a non-causal channel: $D \to M \leftarrow U \to Y$.
That is a collider bias inside the mediator regression. The result is that $\widehat{c'}$ is no longer
the direct effect. The bias has a name: **sequential ignorability violation**.

Let us simulate.

In [ ]:
def make_confounded_mediator(rng, N=4000, a=0.8, b=0.7, c_prime=0.5, gamma=0.6):
    D = rng.binomial(1, 0.5, size=N)
    U = rng.normal(0, 1, size=N)              # unobserved M<->Y confounder
    uM = rng.normal(0, 1, size=N)
    uY = rng.normal(0, 1, size=N)
    M = a * D + gamma * U + uM
    Y = c_prime * D + b * M + gamma * U + uY
    df = pd.DataFrame({"D": D, "M": M, "Y": Y, "U": U})
    df.attrs.update({"a": a, "b": b, "c_prime": c_prime, "gamma": gamma})
    return df

df_conf = make_confounded_mediator(rng)
bk_conf = baron_kenny(df_conf)
truth_total = df_conf.attrs["c_prime"] + df_conf.attrs["a"] * df_conf.attrs["b"]
print(f"true direct  = {df_conf.attrs['c_prime']}")
print(f"true indirect= {df_conf.attrs['a'] * df_conf.attrs['b']}")
print(f"true total   = {truth_total}")
print()
pd.Series(bk_conf)

**The damage.** On the *confounded* DGP, Baron-Kenny's $\widehat{c'}$ overshoots the true direct
effect (because the $D \to M \leftarrow U \to Y$ path adds a positive component when $\gamma > 0$),
and the implied indirect $\widehat a \cdot \widehat b$ also distorts. The *total* effect is still
fine because the $Y \sim D$ regression does not condition on $M$ - so the back-door is closed for the
total. The lesson: **mediation decomposition without sequential ignorability is just plausible-sounding
arithmetic.**

## 4. Sensitivity bound: how much $\gamma$ does it take?

Imai-Keele-Tingley (2010, *Psychological Methods* 15(4):309-334) propose a **sensitivity parameter**
$\rho$: the correlation between the residuals of the mediator equation and the outcome equation. Under
sequential ignorability $\rho = 0$. The larger $|\rho|$, the more $U$ matters. We can sweep $\rho$ and
ask: at what $\rho$ does the indirect effect estimate cross zero?

In [ ]:
def sweep_rho(rho_grid, N=4000, a=0.8, b=0.7, c_prime=0.5, seed=99):
    rng_s = np.random.default_rng(seed)
    records = []
    for rho in rho_grid:
        # Calibrate gamma to deliver the requested rho.
        # Var(resid_M) = gamma^2 + 1; Var(resid_Y) = gamma^2 + 1; Cov = gamma^2
        # rho = gamma^2 / (gamma^2 + 1) => gamma = sqrt(rho / (1-rho)) for rho in (0,1)
        if rho <= 0:
            gamma = 0.0
        elif rho >= 0.95:
            gamma = np.sqrt(0.95 / 0.05)
        else:
            gamma = np.sqrt(rho / (1 - rho))
        D = rng_s.binomial(1, 0.5, size=N)
        U = rng_s.normal(0, 1, size=N)
        uM = rng_s.normal(0, 1, size=N); uY = rng_s.normal(0, 1, size=N)
        M = a * D + gamma * U + uM
        Y = c_prime * D + b * M + gamma * U + uY
        df_s = pd.DataFrame({"D": D, "M": M, "Y": Y})
        bk_s = baron_kenny(df_s)
        records.append({"rho": rho, "indirect_hat": bk_s["indirect_ab"],
                        "direct_hat": bk_s["c_prime_hat"], "total_hat": bk_s["total"],
                        "gamma": gamma})
    return pd.DataFrame(records)

sweep = sweep_rho(np.linspace(0.0, 0.7, 15))
print(f"true indirect = {0.8 * 0.7}")
print(f"true direct   = 0.5")
sweep

In [ ]:
fig, ax = plt.subplots()
ax.plot(sweep["rho"], sweep["indirect_hat"], marker="o", label="indirect (estimated)")
ax.plot(sweep["rho"], sweep["direct_hat"],   marker="s", label="direct (estimated)")
ax.axhline(0.8 * 0.7, ls="--", color="C0", alpha=0.5, label="truth: indirect")
ax.axhline(0.5,       ls="--", color="C1", alpha=0.5, label="truth: direct")
ax.set_xlabel(r"$\rho$ (sensitivity to $M$-$Y$ confounding)")
ax.set_ylabel("Baron-Kenny estimate")
ax.set_title("Bias of mediation decomposition as $M$-$Y$ confounding grows")
ax.legend()
plt.tight_layout(); plt.close(fig)
"plotted"


**Read the plot.** At $\rho = 0$ the estimates land on the dashed truth lines. As $\rho$ grows,
the **direct** effect and the **indirect** effect both drift away from their truths in equal-and-
opposite directions, so the two errors compensate inside the *total* effect. A reviewer who reads only
the total would never notice. A reviewer who reads either the direct or the indirect would draw a
wrong conclusion. The takeaway is not a particular sign of bias - it is that the decomposition is
**uninterpretable** without the sequential-ignorability assumption stated explicitly and bounded.

## 5. The bad-control collider DGP

Now the second failure mode: a variable $C$ that **looks like a clean covariate** but is in fact a
collider on $D$ and an unobserved $U$. We do *not* care about mediation here; we are just trying to
estimate the ATE. Adding $C$ as a "control" creates the bias.

In [ ]:
def make_collider_dgp(rng, N=4000, tau=0.5, alpha_DC=0.7, alpha_UC=0.7, alpha_UY=0.6):
    D = rng.binomial(1, 0.5, size=N)
    U = rng.normal(0, 1, size=N)
    uC = rng.normal(0, 0.5, size=N)
    uY = rng.normal(0, 1, size=N)
    C = alpha_DC * D + alpha_UC * U + uC
    Y = tau * D + alpha_UY * U + uY
    return pd.DataFrame({"D": D, "C": C, "Y": Y}), tau

df_coll, tau_true = make_collider_dgp(rng)
no_ctrl = sm.OLS(df_coll["Y"].values, sm.add_constant(df_coll[["D"]].values)).fit()
ctrl    = sm.OLS(df_coll["Y"].values, sm.add_constant(df_coll[["D","C"]].values)).fit()
print(f"true tau              = {tau_true}")
print(f"OLS  Y ~ D            = {no_ctrl.params[1]:0.3f}")
print(f"OLS  Y ~ D + C (BAD!) = {ctrl.params[1]:0.3f}")

**The damage.** Without the "control" $C$, OLS is unbiased: $D$ is randomized and there is no
back-door. Adding $C$ - because $C$ is a collider on $(D, U)$ and $U$ also affects $Y$ - opens the path
$D \to C \leftarrow U \to Y$ and biases the OLS coefficient on $D$. The direction of the bias depends
on the signs of the path coefficients; here the bias pulls the estimate *toward zero* (the spurious
correlation that conditioning induces between $D$ and $U$ is positive, but the slice of $Y$ that
conditioning attributes to $C$ pulls the $D$ coefficient down). What matters for the paper is not the
sign but the fact that **a control that looked innocent destroyed a clean randomization estimate.**

## 6. A second look at the indirect effect: the "proportion mediated"

A natural summary statistic that appears in many empirical mediation papers is the
**proportion mediated**: indirect / total. On the clean DGP this is $ab / (c' + ab) = 0.56 / 1.06
\approx 0.53$. The mediator carries about half of the total effect. The danger: on the confounded
DGP the same ratio is misleading even when the magnitudes look "fine," because the numerator and
denominator are biased in opposite directions.

In [ ]:
prop_clean = bk["indirect_ab"] / bk["total"]
prop_confounded = bk_conf["indirect_ab"] / bk_conf["total"]
true_prop = (0.8 * 0.7) / (0.5 + 0.8 * 0.7)
print(f"true proportion mediated         = {true_prop:0.3f}")
print(f"clean DGP, Baron-Kenny estimate  = {prop_clean:0.3f}")
print(f"confounded DGP, B-K estimate     = {prop_confounded:0.3f}")

**Read the printout.** On the clean DGP the estimated proportion mediated matches the truth.
On the confounded DGP it drifts away. The drift is small in our calibration but real - and in a
field setting where you do not know the truth, you have no way to detect it without a sensitivity
analysis.

## 7. The DAG litmus test

Both failure modes share a structural fingerprint that you can spot before running any regression. The
question to ask of every candidate control variable $W$ on its way into your specification:

1. **Is $W$ a child of the treatment?** If yes, $W$ is either a mediator or a collider. *Never* a
   clean control.
2. **Does $W$ share an unobserved cause with the outcome?** If yes, conditioning on $W$ opens a
   back-door.
3. **Is $W$ a pre-treatment characteristic?** If yes, $W$ is a candidate clean control.

This is the **pre-treatment rule** that Cinelli, Forney & Pearl (2024, *Sociological Methods & Research*
53(3):1071-1104) make formal. The rule is simple, conservative, and the easiest defense against the
bad-control problem you saw above.

## 8. A reporting template

The minimum honest write-up for a mediation result on observational data:

> **Mediation report.** We decompose the total effect of $D$ on $Y$ into a direct effect (holding
> $M$ fixed) and an indirect effect (through $M$), under the **sequential ignorability** assumption
> (Imai, Keele & Yamamoto 2010): conditional on observed pre-treatment covariates $\mathbf{X}$, (i)
> $D$ is independent of the potential mediators and outcomes, and (ii) $M$ is independent of the
> potential outcomes. We estimate the components using the Baron-Kenny equations, and we report a
> sensitivity bound: across $\rho \in [0, 0.5]$, the indirect effect varies from $X$ to $Y$ and the
> direct effect from $X'$ to $Y'$. The result is robust across this range.

The point of the second sentence is that you have *named* the assumption that lets the decomposition
exist at all. The point of the third sentence is that you have shown how brittle it is.

## 9. Your turn

1. **Negative-$\rho$ world.** Modify the sensitivity sweep to allow $\rho < 0$ (the unobserved $U$
   pushes $M$ up but $Y$ down). Show that the direct-effect bias flips sign.
2. **Pretreatment $M$.** Move the mediator equation to depend only on baseline covariates, not on
   $D$. Re-run Baron-Kenny. What does the indirect effect look like and why?
3. **Collider sanity check.** Re-run the bad-control DGP with `alpha_UY = 0`. Does conditioning on
   $C$ still bias the treatment effect? What's the structural reason?

**Citations.** Baron & Kenny (1986, *Journal of Personality and Social Psychology* 51(6):1173-1182);
Imai, Keele & Yamamoto (2010, *Statistical Science* 25(1):51-71); Imai, Keele & Tingley (2010,
*Psychological Methods* 15(4):309-334); VanderWeele (2015, *Explanation in Causal Inference*, Oxford
University Press); Cinelli, Forney & Pearl (2024, *Sociological Methods & Research* 53(3):1071-1104).